# 12 KV cache、batching、prefill 和 decode

目标：把端到端生成拆成 prefill 和 decode，观察 batch padding、KV cache 和吞吐指标。


## 1. 安装依赖


In [ ]:
from pathlib import Path

base = Path.cwd()
requirements_path = base / "requirements.txt"
advanced_requirements_path = base / "advanced" / "requirements-advanced.txt"

if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

if not advanced_requirements_path.exists():
    if Path("requirements-advanced.txt").exists():
        advanced_requirements_path = Path("requirements-advanced.txt")
    else:
        advanced_requirements_path = Path("../advanced/requirements-advanced.txt")

print("requirements:", requirements_path)
print("advanced requirements:", advanced_requirements_path)
%pip install -r {requirements_path} -r {advanced_requirements_path}


## 2. 模型与工具函数


In [ ]:
import gc
import os
from pathlib import Path
from time import perf_counter

import torch
from modelscope import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE == "modelscope":
        return snapshot_download(model_id)
    return model_id


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def cuda_memory(label=""):
    if not torch.cuda.is_available():
        print(label, "cuda unavailable")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{label} allocated={allocated:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB")


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)
print("cuda =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))
    print("bf16 supported =", torch.cuda.is_bf16_supported())


## 3. 加载模型


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side="left", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
cuda_memory("after load")


## 4. padding side 对 batch 的影响

decoder-only 生成通常推荐 left padding，因为真实 token 会靠近序列末尾。


In [ ]:
texts = ["你好", "请用三句话解释 KV cache 是什么"]

for side in ["left", "right"]:
    tokenizer.padding_side = side
    batch = tokenizer(texts, padding=True, return_tensors="pt")
    print("=" * 40)
    print("padding_side:", side)
    print(batch["input_ids"])
    print(batch["attention_mask"])


## 5. 手动拆 prefill 和 decode

prefill 一次性处理 prompt；decode 每次只喂最新 token，并复用 `past_key_values`。


In [ ]:
tokenizer.padding_side = "left"
messages = [{"role": "user", "content": "用要点解释 prefill、decode 和 KV cache 的关系。"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

if torch.cuda.is_available():
    torch.cuda.synchronize()
start = perf_counter()
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
if torch.cuda.is_available():
    torch.cuda.synchronize()
prefill_seconds = perf_counter() - start

past_key_values = outputs.past_key_values
next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
print("input tokens:", inputs["input_ids"].shape[-1])
print("prefill seconds:", round(prefill_seconds, 4))
print("next token id:", next_token.item())
print("kv cache type:", type(past_key_values).__name__)
if hasattr(past_key_values, "get_seq_length"):
    print("kv cache seq len:", past_key_values.get_seq_length())


In [ ]:
decode_steps = 64
generated = []
attention_mask = inputs["attention_mask"]

if torch.cuda.is_available():
    torch.cuda.synchronize()
start = perf_counter()

with torch.no_grad():
    for _ in range(decode_steps):
        attention_mask = torch.cat(
            [attention_mask, torch.ones((attention_mask.shape[0], 1), device=attention_mask.device, dtype=attention_mask.dtype)],
            dim=-1,
        )
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated.append(next_token)

if torch.cuda.is_available():
    torch.cuda.synchronize()
decode_seconds = perf_counter() - start

new_token_ids = torch.cat(generated, dim=-1)[0]
print("decode tokens:", decode_steps)
print("decode seconds:", round(decode_seconds, 4))
print("decode tokens/s:", round(decode_steps / decode_seconds, 2))
print(tokenizer.decode(new_token_ids, skip_special_tokens=True))
cuda_memory("after manual decode")


## 6. KV cache 显存估算公式

近似公式：`layers * batch * seq_len * kv_heads * head_dim * 2(K,V) * bytes`。GQA 会让 `kv_heads` 小于 `attention_heads`，因此显著降低 KV cache。


In [ ]:
config = model.config
layers = config.num_hidden_layers
attention_heads = config.num_attention_heads
kv_heads = getattr(config, "num_key_value_heads", attention_heads)
head_dim = config.hidden_size // attention_heads


def estimate_kv_cache_gb(batch, seq_len, dtype_bytes=2):
    total_bytes = layers * batch * seq_len * kv_heads * head_dim * 2 * dtype_bytes
    return total_bytes / 1024**3

for seq_len in [512, 2048, 8192, 32768]:
    print(
        f"batch=1 seq_len={seq_len:<5} kv_cache≈{estimate_kv_cache_gb(1, seq_len):.3f}GB "
        f"| batch=8≈{estimate_kv_cache_gb(8, seq_len):.3f}GB"
    )

print("layers:", layers)
print("attention_heads:", attention_heads)
print("kv_heads:", kv_heads)
print("head_dim:", head_dim)


## 面试总结

- prefill 主要和 prompt 长度有关，通常并行度高。
- decode 是逐 token 自回归，延迟和 tokens/s 更关键。
- KV cache 随 batch、上下文长度和层数线性增长，是长上下文/高并发部署的显存核心。
